# Pelajaran 08 - Corak Reka Bentuk Multi-Ejen


## Persediaan


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mengapa Sistem Pelbagai Ejen?

Tugas dunia sebenar seperti perancangan perjalanan melibatkan pelbagai jenis kepakaran — logistik, pengetahuan tempatan, penganggaran, dan banyak lagi. Seorang ejen tunggal yang cuba mengendalikan semuanya dengan cepat menjadi tidak terkawal.

Sistem pelbagai ejen menyelesaikan masalah ini melalui **kepakaran khusus**: setiap ejen menumpukan pada satu bidang kepakaran, menghasilkan keputusan yang lebih berkualiti berbanding seorang generalis. Mereka juga meningkatkan **kebolehskalaan** — anda boleh menambah ejen baru (contohnya, pakar penerbangan, pengkritik restoran) tanpa menulis semula aliran kerja yang sedia ada. Ejen-ejen tersebut berkomposisi melalui saluran yang tersusun, memindahkan konteks dari satu ejen ke ejen yang seterusnya.


## Mencipta Ejen Pakar


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Membina Aliran Kerja Sejajar

`WorkflowBuilder` membolehkan anda menyambungkan ejen ke dalam graf terarah. Di sini kami mencipta saluran dua langkah yang mudah: **TravelPlanner** menyediakan draf jadual perjalanan, kemudian **TravelConcierge** menyemak dan memperbaikinya.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Menambah Lebih Ramai Ejen ke Aliran Kerja

Salah satu kelebihan terbesar corak multi-ejen adalah betapa mudahnya untuk diperluaskan. Di bawah, kami menambah ejen **BudgetReviewer** yang memeriksa rancangan berbanding bajet pengembara, menandakan item yang mungkin mendorong kos melebihi had, dan mencadangkan alternatif penjimatan wang. Aliran kerja kini menjalankan tiga ejen secara berurutan:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

Dalam pelajaran ini anda belajar bagaimana untuk:

1. **Mencipta ejen khusus** — setiap satu dengan peranan tertumpu (perancangan, perkhidmatan, semakan bajet).
2. **Menyambungkan ejen ke dalam aliran kerja berurutan** menggunakan `WorkflowBuilder` dan `add_edge`.
3. **Menstrim output** dari rangkaian ejen pelbagai, mengesan ejen mana yang sedang bercakap.
4. **Memperluaskan aliran kerja** dengan menambah ejen baru ke rantai tanpa mengubah ejen sedia ada.

Corak reka bentuk pelbagai ejen menjadikan setiap ejen mudah sambil menghasilkan keputusan yang lebih kaya dan lebih teliti daripada mana-mana ejen tunggal yang boleh dicapai sendiri.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan penterjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil perhatian bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, penterjemahan profesional oleh manusia adalah disyorkan. Kami tidak bertanggungjawab atas sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
